In [ ]:
!nvidia-smi

Thu May 21 04:20:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -U transformers accelerate soundfile huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 56.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
text_path = "/content/drive/MyDrive/texttospeech.txt"

with open(text_path, "r", encoding="utf-8") as file:
    extracted_text = file.read()

print(extracted_text)

# The Luxury Trap

The rise of farming was a very gradual affair spread over centuries and millennia. A band of *Homo sapiens* gathering mushrooms and nuts and hunting deer and rabbit did not all of a sudden settle in a permanent village, ploughing fields, sowing wheat and carrying water from the river. The change proceeded by stages, each of which involved just a small alteration in daily life.

*Homo sapiens* reached the Middle East around 70,000 years ago. For the next 50,000 years our ancestors flourished there without agriculture. The natural resources of the area were enough to support its human population. In times of plenty people had a few more children, and in times of need a few less. Humans, like many mammals, have hormonal and genetic mechanisms that help control procreation. In good times females reach puberty earlier, and their chances of getting pregnant are a bit higher. In bad times puberty is late and fertility decreases.

To these natural population controls were ad

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Sesame

In [ ]:
import torch
from transformers import CsmForConditionalGeneration, AutoProcessor

model_id = "sesame/csm-1b"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)

model = CsmForConditionalGeneration.from_pretrained(
    model_id,
    device_map=device
)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/538 [00:00<?, ?it/s]

In [ ]:
text = f"[0]{extracted_text}"

inputs = processor(
    text,
    add_special_tokens=True
).to(device)

audio = model.generate(
    **inputs,
    output_audio=True
)

In [ ]:
audio_path = "/content/drive/MyDrive/monreader_sesame_output.wav"

processor.save_audio(
    audio,
    audio_path
)

print("Saved audio to:", audio_path)

Saved audio to: /content/drive/MyDrive/monreader_sesame_output.wav


In [ ]:
from IPython.display import Audio

Audio(audio_path)

## Microsoft Vibevoice

In [ ]:
# Clone the VibeVoice repository
![ -d /content/VibeVoice ] || git clone --quiet --branch main --depth 1 https://github.com/microsoft/VibeVoice.git /content/VibeVoice
print("✅ Cloned VibeVoice repository")

# Install project dependencies
!uv pip --quiet install --system -e /content/VibeVoice[streamingtts]
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
print("✅ Installed dependencies")

# Download model
from huggingface_hub import snapshot_download
snapshot_download("microsoft/VibeVoice-Realtime-0.5B", local_dir="/content/models/VibeVoice-Realtime-0.5B")
print("✅ Downloaded model: microsoft/VibeVoice-Realtime-0.5B")

✅ Cloned VibeVoice repository
✅ Installed dependencies


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

figures/Fig1.png:   0%|          | 0.00/124k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.04G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/360 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Downloaded model: microsoft/VibeVoice-Realtime-0.5B


In [ ]:
import subprocess, re, time, threading, os
from google.colab import drive


print("Loaded text:")
print(extracted_text[:500])

# 5. Start VibeVoice server
srv = subprocess.Popen(
    "python /content/VibeVoice/demo/vibevoice_realtime_demo.py "
    "--model_path /content/models/VibeVoice-Realtime-0.5B "
    "--port 8000",
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True,
)

# 6. Start Cloudflare tunnel
cf = subprocess.Popen(
    "./cloudflared tunnel --url http://localhost:8000 --no-autoupdate",
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True,
)

public_url = None
server_ready = False
url_pattern = re.compile(r"(https://[a-z0-9-]+\.trycloudflare\.com)")

def read_srv():
    global server_ready
    for ln in srv.stdout:
        print("[SERVER]", ln.strip())
        if "Uvicorn running on" in ln or "Running on local URL" in ln:
            server_ready = True

def read_cf():
    global public_url
    for ln in cf.stdout:
        print("[TUNNEL]", ln.strip())
        m = url_pattern.search(ln)
        if m:
            public_url = m.group(1)

threading.Thread(target=read_srv, daemon=True).start()
threading.Thread(target=read_cf, daemon=True).start()

# 7. Wait for public URL
while True:
    if server_ready and public_url:
        print("\n✅ VibeVoice is ready.")
        print(f"Public URL: {public_url}")
        print("\nCopy this text into the VibeVoice input box:")
        print("-" * 60)
        print(extracted_text)
        print("-" * 60)
        break

    time.sleep(0.25)

Loaded text:
# The Luxury Trap

The rise of farming was a very gradual affair spread over centuries and millennia. A band of *Homo sapiens* gathering mushrooms and nuts and hunting deer and rabbit did not all of a sudden settle in a permanent village, ploughing fields, sowing wheat and carrying water from the river. The change proceeded by stages, each of which involved just a small alteration in daily life.

*Homo sapiens* reached the Middle East around 70,000 years ago. For the next 50,000 years our ancest
[TUNNEL] 2026-05-21T05:50:52Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you s

In [ ]:
srv.terminate()
cf.terminate()

print("Stopped VibeVoice server and Cloudflare tunnel.")

Stopped VibeVoice server and Cloudflare tunnel.
